# 02 — Inference on the trained model

Loads the published `best.pt` from the GitHub Release and runs it on validation images and on new
images the model has never seen.

**How to run:** Runtime → **Run all**. A GPU is nice but not required.

**Expected runtime:** ~2–3 min including the install.

> This model is an assistive tool for preliminary screening only. It produces false negatives and
> must not be used as the sole verifier for life-safety decisions. A qualified reviewer checks
> every detection.

## 1. Setup

In [ ]:
!pip install -q ultralytics==8.3.40

import torch, ultralytics
ultralytics.checks()
print("ultralytics:", ultralytics.__version__, "| cuda:", torch.cuda.is_available())

## 2. Configuration

Weights come from a public URL, never from a personal drive — that is what makes this notebook runnable by a stranger.

In [ ]:
WEIGHTS_URL = "FILL: https://github.com/<org>/<repo>/releases/download/v1.0/best.pt"
# classes: helmet, no-helmet, no-vest, person, vest
CONF_THRESHOLD = 0.35    # 0.35 matches the training notebook; lower it to trade precision for recall
                         # on no-helmet (precision 0.947 leaves headroom) — see docs/error_analysis.md
IMGSZ = 640

# A few images the model has never seen. Direct image URLs, or leave empty and upload your own.
NEW_IMAGE_URLS = [
    # "https://.../site_photo_1.jpg",
]

In [ ]:
!wget -q -O /content/best.pt "$WEIGHTS_URL"

from ultralytics import YOLO
model = YOLO("/content/best.pt")
print("classes:", model.names)

## 3. Validation images

A sanity check that the released weights behave as reported. These images were held out of training but the model was tuned against them — section 4 is the honest test.

In [ ]:
import os, glob, zipfile
from google.colab import files
from IPython.display import Image, display

VAL_DIR = "/content/val_images"
os.makedirs(VAL_DIR, exist_ok=True)

if not os.listdir(VAL_DIR):
    print("Upload a handful of validation images (or the dataset zip).")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                z.extractall(VAL_DIR)
        else:
            os.replace(name, os.path.join(VAL_DIR, name))

val_images = [p for p in glob.glob(f"{VAL_DIR}/**/*", recursive=True)
              if p.lower().endswith((".jpg", ".jpeg", ".png"))][:10]

res = model.predict(val_images, conf=CONF_THRESHOLD, imgsz=IMGSZ, save=True,
                    project="/content/runs", name="val_preds")

for p in sorted(glob.glob(f"{res[0].save_dir}/*"))[:5]:
    display(Image(filename=p, width=560))

## 4. New images

The real demonstration: photographs from outside the dataset. If it works here, there is a product.

In [ ]:
NEW_DIR = "/content/new_images"
os.makedirs(NEW_DIR, exist_ok=True)

for i, url in enumerate(NEW_IMAGE_URLS):
    !wget -q -O "{NEW_DIR}/new_{i+1:02d}.jpg" "{url}"

if not os.listdir(NEW_DIR):
    print("Upload 5 new images the model has never seen.")
    uploaded = files.upload()
    for name in uploaded:
        os.replace(name, os.path.join(NEW_DIR, name))

new_images = sorted(glob.glob(f"{NEW_DIR}/*"))
res_new = model.predict(new_images, conf=CONF_THRESHOLD, imgsz=IMGSZ, save=True,
                        project="/content/runs", name="new_preds")

for r, p in zip(res_new, sorted(glob.glob(f"{res_new[0].save_dir}/*"))):
    counts = {}
    for c in r.boxes.cls.tolist():
        counts[model.names[int(c)]] = counts.get(model.names[int(c)], 0) + 1
    print(os.path.basename(p), "->", counts or "no detections")
    display(Image(filename=p, width=560))

## 5. Confidence sweep

The threshold is a business decision, not a default. This shows what recall-first vs precision-first actually costs on your own images.

In [ ]:
for t in (0.10, 0.25, 0.50, 0.75):
    r = model.predict(new_images, conf=t, imgsz=IMGSZ, verbose=False)
    total = sum(len(x.boxes) for x in r)
    print(f"conf {t:>4}: {total:>3} detections across {len(new_images)} images")

## 6. Download the evidence

In [ ]:
import shutil
shutil.make_archive("/content/inference_evidence", "zip", "/content/runs")
files.download("/content/inference_evidence.zip")

## How to read these results

- A missed instance is a **false negative** — the expensive error for safety screening.
- A box on nothing is a **false positive** — the expensive error when detections trigger cost.
- Right object with the wrong label is **class confusion**; right place with a sloppy box is a
  **localisation error**.

Three of each go in [`docs/error_analysis.md`](../docs/error_analysis.md) with a hypothesis for
*why*. A well-explained failure is worth more than a hidden one.